In [ ]:
import numpy as np
import pandas as pd
import ccxt
from ripser import ripser
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import time
import asyncio
import configparser
import hashlib
import hmac
import json
import threading
import queue
from dataclasses import dataclass
from decimal import Decimal, ROUND_DOWN

import numpy as np
import requests
import websockets

import matplotlib
matplotlib.use("TkAgg")
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation


## Config ##


In [ ]:
INTERVAL = "1m"
TICKERS = ['BTCUSDT', 'ETHUSDT', 'BNBUSDT', 'SOLUSDT', 'XRPUSDT']  # expand to 40+
symbol = TICKERS[0] # e.g. 'BTCUSDT'

TESTNET_REST = "https://testnet.binance.vision"
TESTNET_WS = f"wss://stream.testnet.binance.vision/ws/{symbol.lower()}@kline_{INTERVAL}"

PRELOAD_LIMIT = 200
WINDOW = 250

# Strategie: SMA Cross + RSI Filter
SMA_SHORT = 5
SMA_LONG = 20
RSI_PERIOD = 14
RSI_BUY_MAX = 75
RSI_SELL_MIN = 25

# Trading (Testnet)
TRADE_ENABLED = True
USDT_PER_BUY = 100.0
RECV_WINDOW = 10000  # vorher 5000, etwas höher = weniger -1021 Stress

# Plot-Update: jede Sekunde checken, gezeichnet wird nur wenn neue Kerze da ist
PLOT_UPDATE_MS = 1000

# Balances/Prints (real für Testnet Guthaben)
BALANCE_EVERY_N_CANDLES = 1
PRINT_EVERY_CLOSED_CANDLE = True

# Portfolio-Simulation (Performance-Visualisierung)
START_USDT_SIM = 1000.0
BUY_USDT_SIM = 100.0


# ============================================================
# BINANCE AUTH / REST + TIME OFFSET
# ============================================================
@dataclass
class BinanceKeys:
    api_key: str
    secret_key: str


# globaler Zeit-Offset (Serverzeit - lokale Zeit) in ms
TIME_OFFSET_MS = 0

NameError: name 'dataclass' is not defined

In [ ]:
def load_keys(cfg_path: str = "secret.cfg") -> BinanceKeys:
    cfg = configparser.ConfigParser()
    if not cfg.read(cfg_path):
        raise FileNotFoundError(f"Konnte {cfg_path} nicht lesen.")
    return BinanceKeys(
        api_key=cfg["BINANCE"]["TEST_API_KEY"].strip(),
        secret_key=cfg["BINANCE"]["TEST_SECRET_KEY"].strip()
    )


def _local_ms() -> int:
    return int(time.time() * 1000)


def _ts_ms() -> int:
    # Server synchronisiert: lokale Zeit + Offset
    return _local_ms() + TIME_OFFSET_MS


def _sign(secret_key: str, query_string: str) -> str:
    return hmac.new(
        secret_key.encode("utf-8"),
        query_string.encode("utf-8"),
        hashlib.sha256
    ).hexdigest()


def public_get(path: str, params: dict | None = None):
    url = f"{TESTNET_REST}{path}"
    r = requests.get(url, params=params, timeout=20)
    if r.status_code >= 400:
        raise RuntimeError(f"Public Fehler {r.status_code}: {r.text}")
    return r.json()


def sync_server_time_offset():
    """
    Holt die Binance-Serverzeit und berechnet den Offset.
    Offset = serverTime - localTime
    """
    global TIME_OFFSET_MS
    try:
        server = public_get("/api/v3/time")
        server_ms = int(server["serverTime"])
        local_ms = _local_ms()
        TIME_OFFSET_MS = server_ms - local_ms
        print(f"[TIME] Offset synchronisiert: {TIME_OFFSET_MS} ms (server-local)")
    except Exception as e:
        print(f"[TIME] Konnte Offset nicht synchronisieren: {e}")


def signed_request(keys: BinanceKeys, method: str, path: str, params: dict):
    """
    Signierte REST Requests mit Zeitoffset-Fix.
    Falls -1021 kommt, wird Offset neu synchronisiert und 1x retry gemacht.
    """
    params = dict(params or {})
    params["timestamp"] = _ts_ms()
    params["recvWindow"] = RECV_WINDOW

    query = "&".join([f"{k}={params[k]}" for k in sorted(params.keys())])
    signature = _sign(keys.secret_key, query)

    url = f"{TESTNET_REST}{path}?{query}&signature={signature}"
    headers = {"X-MBX-APIKEY": keys.api_key}

    r = requests.request(method=method, url=url, headers=headers, timeout=20)

    if r.status_code >= 400:
        # Versuch: wenn Timestamp-Problem, Offset neu sync + retry
        try:
            err = r.json()
        except Exception:
            err = {"raw": r.text}

        if isinstance(err, dict) and err.get("code") == -1021:
            print("[TIME] -1021 Timestamp-Fehler -> resync und retry")
            sync_server_time_offset()

            params["timestamp"] = _ts_ms()
            query = "&".join([f"{k}={params[k]}" for k in sorted(params.keys())])
            signature = _sign(keys.secret_key, query)
            url = f"{TESTNET_REST}{path}?{query}&signature={signature}"

            r2 = requests.request(method=method, url=url, headers=headers, timeout=20)
            if r2.status_code >= 400:
                raise RuntimeError(f"Signed Fehler {r2.status_code}: {r2.text}")
            return r2.json()

        raise RuntimeError(f"Signed Fehler {r.status_code}: {r.text}")

    return r.json()

In [ ]:
# 1. Fetch data
# ======================
exchange = ccxt.binance()
tickers = ['BTC/USDT', 'ETH/USDT', 'BNB/USDT', 'SOL/USDT', 'XRP/USDT']  # expand to 40+

def fetch_returns(tickers, days=500):
    dfs = []
    for symbol in tickers:
        ohlcv = exchange.fetch_ohlcv(symbol, '1d', limit=days)
        df = pd.DataFrame(ohlcv, columns=['timestamp','open','high','low','close','volume'])
        df['date'] = pd.to_datetime(df['timestamp'], unit='ms')
        df.set_index('date', inplace=True)
        df['returns'] = df['close'].pct_change()
        dfs.append(df['returns'])
    data = pd.concat(dfs, axis=1)
    data.columns = tickers
    data.dropna(inplace=True)
    return data

returns = fetch_returns(tickers, days=500)
